# RoadWatch — Train Vietnamese traffic-sign detector (YOLO11n) for CPU inference at 416

Run on **Google Colab or Kaggle with a GPU** (Runtime → Change runtime type → GPU). ~1–2 h on a T4.

Output: `best.pt` (+ a 416 ONNX for a quick check). On the laptop, export and register it with:

```bash
.venv-export/bin/python scripts/export_yolo_onnx.py --weights best.pt --imgsz 416 --name vn_signs_416 \
    --source "VR-TSD fine-tune of yolo11n.pt (notebooks/train_signs.ipynb)"
```

RoadWatch automatically prefers `models/vn_signs_416.onnx` over the upstream 768 model.

**Why retrain?** The upstream model only ships a fixed 768×768 ONNX (~210 ms per run on the laptop CPU when
running alongside the other models). A 416 model is ~3× cheaper.

## 1. Setup

In [ ]:
!nvidia-smi -L
%pip install -q "ultralytics>=8.3" roboflow onnx onnxslim
import ultralytics; ultralytics.checks()

## 2. Download VR-TSD from Roboflow

Dataset page: https://universe.roboflow.com/vietnam-traffic-sign-recognition-benchmark/vr-tsd

1. Sign in to Roboflow (free), open the dataset page, pick a version and note its number.
2. Put your API key in Colab **Secrets** as `ROBOFLOW_API_KEY` (key icon on the left), or paste it when prompted.
   Do not hard-code the key in the notebook.

In [ ]:
import os, getpass
try:
    from google.colab import userdata
    API_KEY = userdata.get("ROBOFLOW_API_KEY")
except Exception:
    API_KEY = os.environ.get("ROBOFLOW_API_KEY") or getpass.getpass("Roboflow API key: ")

WORKSPACE = "vietnam-traffic-sign-recognition-benchmark"
PROJECT = "vr-tsd"
VERSION = 1  # <-- set to the version number shown on the dataset page

from roboflow import Roboflow
rf = Roboflow(api_key=API_KEY)
dataset = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download("yolov11")
DATA_YAML = os.path.join(dataset.location, "data.yaml")
print(DATA_YAML)

## 3. Check the class list

RoadWatch matches classes **by name** using `models/sign_classes.json`, so the order may differ,
but every class name below must exist there. Unknown names mean `sign_classes.json` needs new entries.

In [ ]:
EXPECTED = [
 "102-cam-di-nguoc-chieu",
 "103a-cam-oto",
 "103b-cam-oto-re-phai",
 "103c-cam-oto-re-trai",
 "106-cam-oto-tai",
 "107-cam-oto-khach-va-oto-tai",
 "123a-cam-re-trai",
 "123b-cam-re-phai",
 "124a-cam-quay-dau-xe",
 "124b-cam-oto-quay-dau-xe",
 "124c-cam-re-trai-va-cam-quay-dau-xe",
 "125-cam-vuot",
 "127-toc-do-toi-da-100",
 "127-toc-do-toi-da-20",
 "127-toc-do-toi-da-30",
 "127-toc-do-toi-da-40",
 "127-toc-do-toi-da-50",
 "127-toc-do-toi-da-60",
 "127-toc-do-toi-da-70",
 "127-toc-do-toi-da-80",
 "127-toc-do-toi-da-90",
 "128-cam-su-dung-coi",
 "130-cam-dung-va-do-xe",
 "131a-cam-do-xe",
 "131b-cam-do-xe-ngay-le",
 "131c-cam-do-xe-ngay-chan",
 "133-het-cam-vuot",
 "134-het-han-che-toc-do-toi-da",
 "135-het-tat-ca-cac-lenh-cam",
 "136-cam-di-thang",
 "137-cam-re-trai-va-re-phai",
 "201-cho-ngoat-nguy-hiem",
 "203-duong-bi-hep",
 "205-duong-giao-nhau",
 "207-giao-nhau-voi-duong-khong-uu-tien",
 "208-giao-nhau-voi-duong-uu-tien",
 "209-giao-nhau-co-tin-hieu-den",
 "221-duong-go-ghe",
 "224-duong-nguoi-di-bo-cat-ngang",
 "225-chu-y-tre-em",
 "227-cong-truong",
 "245-di-cham",
 "301a-cac-xe-chi-duoc-di-thang",
 "301b-cac-xe-chi-duoc-re-phai",
 "301c-cac-xe-chi-duoc-re-trai",
 "301d-cac-xe-chi-duoc-re-phai-2",
 "301e-cac-xe-chi-duoc-re-trai-2",
 "301h-cac-xe-chi-duoc-di-thang-va-re-trai",
 "302a-huong-phai-di-vong-chuong-ngai-vat-1",
 "303-noi-giao-nhau-chay-theo-vong-xuyen",
 "306-toc-do-toi-thieu-60",
 "407a-duong-mot-chieu",
 "420-bat-dau-khu-dong-dan-cu",
 "421-ket-thuc-khu-dong-dan-cu",
 "423a-duong-nguoi-di-bo-sang-ngang-1",
 "437-bat-dau-duong-cao-toc",
 "438-ket-thuc-duong-cao-toc",
 "442-cho"
]

import yaml
with open(DATA_YAML) as f:
    data = yaml.safe_load(f)
ds_names = data["names"] if isinstance(data["names"], list) else [data["names"][i] for i in sorted(data["names"])]
print(len(ds_names), "classes in dataset")
unknown = [n for n in ds_names if n not in EXPECTED]
missing = [n for n in EXPECTED if n not in ds_names]
print("Not in sign_classes.json:", unknown or "none")
print("Expected but absent:", missing or "none")

In [ ]:
# Instances per class in the training split (helps spot rare classes).
import glob, collections
counts = collections.Counter()
for p in glob.glob(os.path.join(dataset.location, "train", "labels", "*.txt")):
    for line in open(p):
        if line.strip():
            counts[int(line.split()[0])] += 1
for i, n in enumerate(ds_names):
    flag = "  <-- few samples" if counts[i] < 50 else ""
    print(f"{i:2d} {n:45s} {counts[i]:5d}{flag}")

## 4. Train

- Start from COCO-pretrained `yolo11n.pt`, `imgsz=640`, ~100 epochs with early stopping.
- **`fliplr=0`**: horizontal flips would turn *no left turn* into *no right turn* (and similar pairs),
  silently corrupting labels.
- Light photometric augmentation for Vietnamese conditions (brightness/haze). Ultralytics also applies
  blur/median-blur/CLAHE through albumentations when it is installed (it is on Colab).

In [ ]:
from ultralytics import YOLO

TRAIN_IMGSZ = 640
model = YOLO("yolo11n.pt")
results = model.train(
    data=DATA_YAML,
    imgsz=TRAIN_IMGSZ,
    epochs=100,
    patience=30,
    batch=-1,          # auto batch size for the GPU
    optimizer="AdamW",
    lr0=0.001,
    cos_lr=True,
    fliplr=0.0,        # keep left/right-specific signs correct
    degrees=5.0,
    translate=0.1,
    scale=0.5,
    hsv_h=0.015, hsv_s=0.6, hsv_v=0.5,
    mosaic=1.0,
    close_mosaic=10,
    project="runs", name="vn_signs", exist_ok=True,
)
BEST = "runs/vn_signs/weights/best.pt"

## 5. Evaluate at the deployment size

RoadWatch runs this model at **416** on the CPU, so the 416 numbers are the ones that matter.
If mAP50 at 416 is much lower than at 640 (small, distant signs), consider retraining with `TRAIN_IMGSZ = 416`
or exporting at 480 instead.

In [ ]:
best = YOLO(BEST)
for sz in (640, 416):
    m = best.val(data=DATA_YAML, imgsz=sz, split="val", plots=False, verbose=False)
    print(f"imgsz {sz}: mAP50 {m.box.map50:.4f}  mAP50-95 {m.box.map:.4f}  P {m.box.mp:.4f}  R {m.box.mr:.4f}")

In [ ]:
# Per-class AP50 at 416 (lowest first).
m = best.val(data=DATA_YAML, imgsz=416, split="val", plots=False, verbose=False)
per_class = sorted(zip(m.box.ap_class_index, m.box.ap50), key=lambda t: t[1])
for idx, ap in per_class:
    print(f"{best.names[int(idx)]:45s} AP50 {ap:.3f}")
print("\nClass list:", [best.names[i] for i in sorted(best.names)])

## 6. Export a 416 ONNX (quick check) and download

In [ ]:
onnx_path = best.export(format="onnx", imgsz=416, simplify=True, dynamic=False, opset=12, nms=False)
print(onnx_path)

import shutil
shutil.copy(BEST, "vn_signs_best.pt")
try:
    from google.colab import files
    files.download("vn_signs_best.pt")
except Exception:
    print("Download vn_signs_best.pt from the file browser (Kaggle: /kaggle/working).")

## 7. On the laptop

```bash
cp ~/Downloads/vn_signs_best.pt models/
.venv-export/bin/python scripts/export_yolo_onnx.py --weights models/vn_signs_best.pt --imgsz 416 \
    --name vn_signs_416 --source "VR-TSD fine-tune of yolo11n.pt (notebooks/train_signs.ipynb)"
.venv/bin/python -m pytest -q tests/test_signs.py   # checks the class names against sign_classes.json
./run_gui.sh
```